## Sound Life Data Retrieval from CELLxGENE Census

**Author:** Lucas Graybuck, Ph.D.  
**Date:** 2026-02-02

This notebook demonstrates how to retrieve specific subsets of data from the Sound Life dataset using the `cellxgene_census` package for Python.

`cellxgene_census` provides and API to datasets stored in the CELLxGENE Collection database. This guide was written with help from the CZ CELLxGENE Discover Census documentation, which is [available here](https://chanzuckerberg.github.io/cellxgene-census/index.html).

### Import packages

In [1]:
import cellxgene_census # The CELLxGENE census API
import numpy as np      # Numeric data handling

import session_info     # Display detailed package/environment information

### Open connection to Census

For the sake of reproducibility, we'll connect to census version 2025-11-08, which was current at the time of creation of this notebook.

In [2]:
census = cellxgene_census.open_soma(
    census_version = '2025-11-08'
)

### Dataset Lookup

First, we'll need to obtain all of the dataset IDs for the Sound Life cohort. We can find these using the Collection UUID, which you can find in the URL of our [Sound Life Collection page](https://cellxgene.cziscience.com/collections/e9360edf-b0b7-4e01-bce8-e596814f13e7)

In [3]:
collection_id = 'e9360edf-b0b7-4e01-bce8-e596814f13e7'

The census datasets can be obtained with the following:

In [4]:
census_datasets = census["census_info"]["datasets"].read().concat().to_pandas()

We'll then filter to our collection:

In [5]:
soundlife_datasets = census_datasets[census_datasets['collection_id'] == collection_id]

In [6]:
soundlife_datasets.shape

(8, 11)

There are 8 datasets from the Sound Life cohort, each from a specific combination, of age group, sex, and CMV infection status.

In [7]:
soundlife_datasets['dataset_title']

1734    Sound Life Young Adult Females, CMV-positive
1735      Sound Life Young Adult Males, CMV-positive
1744      Sound Life Older Adult Males, CMV-positive
1746    Sound Life Older Adult Females, CMV-negative
1771      Sound Life Young Adult Males, CMV-negative
1775      Sound Life Older Adult Males, CMV-negative
1796    Sound Life Older Adult Females, CMV-positive
1798    Sound Life Young Adult Females, CMV-negative
Name: dataset_title, dtype: object

### Query for observations/cell metadata

Next, we'll query the CELLxGENE Census API for cells from Sound Life. We'll show you how to select cells from specific subjects and cell types.

It's a good idea to run queries for observations first, before retrieving data. This will ensure we don't bite off more data than we're prepared to process.

First, we'll need to concatenate the datasets and format them for the Census query:

In [8]:
dataset_ids = "','".join(soundlife_datasets['dataset_id'])

In [9]:
dataset_filter = f"dataset_id in ['{dataset_ids}']"

First, let's get all of the available cells so we can get some information about the dataset.

In [10]:
all_obs = cellxgene_census.get_obs(
    census,
    organism = 'Homo sapiens',
    value_filter = dataset_filter
)

In [11]:
all_obs.shape

(13789548, 28)

With over 13 million cells available, that may be more than we want to download.

In [12]:
all_obs.columns

Index(['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id',
       'cell_type', 'cell_type_ontology_term_id', 'development_stage',
       'development_stage_ontology_term_id', 'disease',
       'disease_ontology_term_id', 'donor_id', 'is_primary_data',
       'observation_joinid', 'self_reported_ethnicity',
       'self_reported_ethnicity_ontology_term_id', 'sex',
       'sex_ontology_term_id', 'suspension_type', 'tissue',
       'tissue_ontology_term_id', 'tissue_type', 'tissue_general',
       'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz',
       'raw_variance_nnz', 'n_measured_vars'],
      dtype='object')

We can make filters for specific subsets of subjects or cells. These filters can use any of the CELLxGENE Standard Categories. A few that are relevant to our dataset are:  
- cell_type
- development_stage (i.e. age)
- self_reported_ethnicity
- sex

#### Filtering examples

For example, if we want just NK cells from 36 year-old Female subjects, we can use the following set of filters:

In [13]:
sex_filter = "sex == 'female'"
age_filter = "development_stage = '36-year-old-stage'"
cell_type_filter = "cell_type == 'natural killer cell'"

Finally, we can combine these with the dataset filter to narrow down our search by assembling the strings with ' and ' as a separator.

In [14]:
obs_filter = ' and '.join([dataset_filter, sex_filter, cell_type_filter])

In [15]:
obs_filter

"dataset_id in ['0a8e3443-c3e2-4918-84db-0495657d9175','5fde5c9c-5b1e-4df5-982a-3f8e7635161f','0c0c49a2-b96b-432f-98a4-1decf9929149','3cb646ab-d411-4e5b-89eb-082a786c953b','420d0f5d-b7ba-4b1e-88d0-44e9f733febd','47e00f43-6176-44a5-b4d1-f88a1920fcab','aa32098f-f7b0-43f2-a00a-601d190aae94','46104f0b-9af5-466a-ae0f-56b8dc1969a2'] and sex == 'female' and cell_type == 'natural killer cell'"

In [16]:
obs_result = cellxgene_census.get_obs(
    census,
    organism = 'Homo sapiens',
    value_filter = obs_filter
)

In [17]:
obs_result.shape

(9874, 28)

To see all cell types and ages, we can get unique values from the `all_obs` table:

In [18]:
cell_type_counts = all_obs['cell_type'].value_counts()
cell_type_counts = cell_type_counts[cell_type_counts > 0]
cell_type_counts

cell_type
naive thymus-derived CD4-positive, alpha-beta T cell     2839099
CD14-positive, CD16-negative classical monocyte          2052523
central memory CD4-positive, alpha-beta T cell           1358321
effector memory CD8-positive, alpha-beta T cell          1242272
effector memory CD4-positive, alpha-beta T cell          1091711
CD16-positive, CD56-dim natural killer cell, human       1008708
naive thymus-derived CD8-positive, alpha-beta T cell      773385
naive B cell                                              668099
memory B cell                                             362092
mucosal invariant T cell                                  357489
CD14-low, CD16-positive monocyte                          327638
mature gamma-delta T cell                                 297338
central memory CD8-positive, alpha-beta T cell            220612
CD4-positive, alpha-beta memory T cell                    190467
memory regulatory T cell                                  142626
naive regulator

In [19]:
age_counts = all_obs['development_stage'].value_counts()
age_counts = age_counts[age_counts > 0]
age_counts = age_counts.reset_index().sort_values('development_stage')
age_counts

,development_stage,count
24,25-year-old stage,67870
20,26-year-old stage,302459
18,27-year-old stage,392208
15,28-year-old stage,478005
2,29-year-old stage,825891
5,30-year-old stage,745613
4,31-year-old stage,775480
3,32-year-old stage,782428
12,33-year-old stage,632042
16,34-year-old stage,444589


Now that we know they're an available cell type, we can easily grab plasma cells from all male subjects:

In [20]:
sex_filter = "sex == 'male'"
cell_type_filter = "cell_type == 'plasma cell'"

In [21]:
obs_filter = ' and '.join([dataset_filter, sex_filter, cell_type_filter])

In [22]:
obs_filter

"dataset_id in ['0a8e3443-c3e2-4918-84db-0495657d9175','5fde5c9c-5b1e-4df5-982a-3f8e7635161f','0c0c49a2-b96b-432f-98a4-1decf9929149','3cb646ab-d411-4e5b-89eb-082a786c953b','420d0f5d-b7ba-4b1e-88d0-44e9f733febd','47e00f43-6176-44a5-b4d1-f88a1920fcab','aa32098f-f7b0-43f2-a00a-601d190aae94','46104f0b-9af5-466a-ae0f-56b8dc1969a2'] and sex == 'male' and cell_type == 'plasma cell'"

In [23]:
obs_result = cellxgene_census.get_obs(
    census,
    organism = 'Homo sapiens',
    value_filter = obs_filter
)

In [24]:
obs_result.shape

(6732, 28)

It's a little trickier to filter by age, but we can do so using a range and the development_stage column. 

Here's an example of how to retrieve memory B cell from subjects between 60 and 70 years old:

In [25]:
ages = range(60,71)
age_list = [f'{age}-year-old stage' for age in ages]
age_filter = "development_stage in ['" + "','".join(age_list) + "']"

cell_type_filter = "cell_type == 'memory B cell'"

In [26]:
obs_filter = ' and '.join([dataset_filter, age_filter, cell_type_filter])

In [27]:
obs_filter

"dataset_id in ['0a8e3443-c3e2-4918-84db-0495657d9175','5fde5c9c-5b1e-4df5-982a-3f8e7635161f','0c0c49a2-b96b-432f-98a4-1decf9929149','3cb646ab-d411-4e5b-89eb-082a786c953b','420d0f5d-b7ba-4b1e-88d0-44e9f733febd','47e00f43-6176-44a5-b4d1-f88a1920fcab','aa32098f-f7b0-43f2-a00a-601d190aae94','46104f0b-9af5-466a-ae0f-56b8dc1969a2'] and development_stage in ['60-year-old stage','61-year-old stage','62-year-old stage','63-year-old stage','64-year-old stage','65-year-old stage','66-year-old stage','67-year-old stage','68-year-old stage','69-year-old stage','70-year-old stage'] and cell_type == 'memory B cell'"

In [28]:
obs_result = cellxgene_census.get_obs(
    census,
    organism = 'Homo sapiens',
    value_filter = obs_filter
)

In [29]:
obs_result.shape

(84821, 28)

### Downloading data

We can use the same filters to download data directly into memory as an AnnData object.

We just need to change the `value_filter` parameter to `obs_value_filter` to specify that we're filtering on cells.

In [30]:
adata = cellxgene_census.get_anndata(
    census,
    organism = 'Homo sapiens',
    obs_value_filter = obs_filter
)

In [31]:
adata

AnnData object with n_obs × n_vars = 84821 × 61497
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'

#### Filtering for measured genes

There are some genes/features here that we didn't actually count in our pipeline. We'll use the presence matrix to get the features we measured in our study, and drop those that weren't used.

For this, we'll need the `soma_joinid` values from our `soundlife_datasets` data frame, above. Since our data all used the same features, we just need the first joinid.

In [32]:
join_id = soundlife_datasets['soma_joinid'].iloc[0]

In [33]:
presence_matrix = cellxgene_census.get_presence_matrix(
    census, 
    organism = "Homo sapiens", 
    measurement_name = "RNA"
)

presence_matrix

<Compressed Sparse Row sparse matrix of dtype 'uint8'
	with 39407438 stored elements and shape (1845, 61497)>

Let's select the row corresponding to our join_id value, then convert to a coordinate matrix and get column values, which identify the present features.

In [34]:
presence_matrix = presence_matrix[join_id,:]
keep_var = presence_matrix.tocoo().col

In [35]:
adata = adata[:,keep_var].copy()

In [36]:
adata.shape

(84821, 32357)

In [37]:
adata.var

,soma_joinid,feature_id,feature_name,feature_type,feature_length,nnz,n_measured_obs
0,0,ENSG00000237491,LINC01409,lncRNA,1089,12566788,142365045
1,1,ENSG00000188976,NOC2L,protein_coding,1244,26123981,158649077
2,2,ENSG00000187642,PERM1,protein_coding,2765,824852,146911882
3,3,ENSG00000272512,ENSG00000272512,lncRNA,2086,1190083,141564179
4,4,ENSG00000188290,HES4,protein_coding,961,24610287,158630852
...,...,...,...,...,...,...,...
36154,36154,ENSG00000288642,CDR1,lncRNA,789,1782898,93128557
36169,36169,ENSG00000288709,F8A2,protein_coding,1707,47610,69822353
36185,36185,ENSG00000288596,C8orf44,lncRNA,665,7107396,59300004
59073,59073,ENSG00000290791,ENSG00000290791,lncRNA,656,219417,23294116


#### Updating value type
By default, `cellxgene_census` returns values as floats, but our raw data are counts.

To save storage space, we can convert these to 16-bit integers before saving a file.

In [38]:
adata.X = adata.X.astype(np.uint16)

Then, we can save this file for later use:

In [39]:
adata.write_h5ad('soundlife_memory-B-cell_age-60-70.h5ad')

## Citation Information

If you utilize the data from the Sound Life cohort please cite these two references:

Gong Q, Sharma M, Kuan EL, Glass MC, Chander A, Singh M, et al. Longitudinal Multi-omic Immune Profiling Reveals Age-Related Immune Cell Dynamics in Healthy Adults. bioRxiv. 2024. p. 2024.09.10.612119. doi:10.1101/2024.09.10.612119

Gustafson CE, Skene PJ, Goldrath AW, Li X-J, Torgerson TR, Becker LA, et al. Longitudinal Dynamics of Health and Age. In: Human Immune System Explorer. doi:10.57785/bn17-3h16

## SessionInfo

In [40]:
session_info.show()

/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/session_info/main.py:213: UserWarning: The '__version__' attribute is deprecated and will be removed in MarkupSafe 3.1. Use feature detection, or `importlib.metadata.version("markupsafe")`, instead.
  mod_version = _find_version(mod.__version__)
